---
title: "DRG Cleaning"

author: "Carlos Resurreccion"

date: "2024-07-01"

---

## Parameters

In [1]:
# IMPORTANT PARAMETERS:
year_to_load <- "2018" # Which claims year to load # TODO: maybe add a script that loops through all claims?
split_parts <- 5 # How many (integer) parts to split the 12+m row claims file into
end_nrow <- 10 # How many rows/entries to show in summary tables

# Input:
to_read <- FALSE # TODO: Deprecated, used to be whether to forcibly read the whole file again instead of using the split parts created even if available
to_split <- TRUE # TODO: Deprecated, only used when to_sample is TRUE # Whether to split into split_parts parts (i.e. to fit in 32gb RAM).
to_sample <- TRUE # Whether to sample each split_parts part by sample_size_divisor (useful when iterating through code runs in quick succession)
sample_size_divisor <- 125 # Sample size divisor: Formula for sample size is total_rows / split_parts / sample_size_divisor. Choose between 5, 25, and 125

# Output:
to_write <- TRUE # Whether to write out intermediate files and caches (i.e. part files, sample files). TODO: upload to BQ as well
to_group <- TRUE # Whether to export for the batch grouper or not

# Debug:
to_debug <- FALSE # whether to print debug statements
to_profvis <- FALSE # Conduct runtime duration analysis via profvis or not
to_view_checks <- TRUE # Whether to view checks and print statements
to_view_checks_parallelized <- FALSE # Whether to view intermediate per part/chunk checks and print statements (not consolidated) when parallelized
to_parallel <- TRUE # Whether to parallelize each split_parts part into availableCores() - 1 chunks. Cuts down processing time from 120min to 15min.
to_split_read <- FALSE # WARNING: TRUE uses a lot of memory!!
tmp_nrow <- Inf # Per part/chunk end_nrow (leave at Inf)

drop_cols <- c( # Which columns to drop
  paste0("ICDCODE", 13:14), # Start
  "ICCODED15", # note that ICDCODE15 is misspelled as ICCODED15 in all claims
  paste0("ICDCODE", 16:170) # Continuation
)

seed <- 123 # Seed for reproducibility (Important for stuff like randomly choosing a pdx among multiple possible options)
set.seed(seed) # Setting the seed
global_seed <- seed # global_seed for future_lapply parts for parallelized operations

ram_size <- 32 # Input virtual or physical machine's RAM size here
ram_buffer <- 0.05 # How much of a buffer to leave for the OS
ram_limit <- (1 - ram_buffer) * (ram_size) * (1024^3) # Compute ram_limit in bytes

# Allowing each future_lapply session to use more memory
options(future.globals.maxSize = ram_limit)

cat(sprintf("RAM usage allowed: %.1f GB", ram_limit / (1024^3)))

if (!split_parts == as.integer(split_parts) || split_parts <= 1) stop("ERROR: split_parts must be an integer greater than or equal to 2!")
if (ram_size <= 64 && split_parts <= 2) stop("Please set split_parts to at least 3 for 64 GB machines or it will likely crash")
if (ram_size <= 32 && split_parts <= 4) stop("Please set split_parts to at least 5 for 32 GB machines or it will likely crash")


RAM usage allowed: 30.4 GB

## Load Required Libraries & Initial Functions

In [2]:
options(verbose = FALSE) # Hide verbose output for script and library loading
options(warn = -1) # Hide warnings for script sourcing and library loading
library(here) # Library here() so scripts can be loaded


here() starts at /home/jupyter/drg-pipeline



In [3]:
scripts <- list( # List of scripts to source
  libraries = "00_libraries.R",
  formats = "01_data-formats.R",
  paths = "02_file-paths.R",
  general = "03_general-functions.R",
  clean = "04a_clean-data-functions.R",
  chunk = "04b_chunk-functions.R",
  part = "04c_part-functions.R",
  io = "05_io-functions.R",
  icd = "06_icd-functions.R",
  rvs = "07_rvs-functions.R",
  pdx = "08_pdx-functions.R",
  grouper = "09_grouper-functions.R",
  timing = "10_timing-functions.R",
  debug = "11_debug-functions.R",
  summary = "12_summary-functions.R"
)

# Loop to source above scripts
for (script in scripts) source(here("data-cleaning/r_scripts", script))


Loading required package: data.table

Loading required package: tictoc


Attaching package: ‘tictoc’


The following object is masked from ‘package:data.table’:

    shift


Loading required package: stringr

Loading required package: stringi

Loading required package: lubridate


Attaching package: ‘lubridate’


The following objects are masked from ‘package:data.table’:

    hour, isoweek, mday, minute, month, quarter, second, wday, week,
    yday, year


The following objects are masked from ‘package:base’:

    date, intersect, setdiff, union


Loading required package: docstring


Attaching package: ‘docstring’


The following object is masked from ‘package:utils’:

    ?


Loading required package: profvis

Loading required package: hash

hash-2.2.6.3 provided by Decision Patterns



Attaching package: ‘hash’


The following object is masked from ‘package:tictoc’:

    clear


The following object is masked from ‘package:data.table’:

    copy


Loading required package: future



Total Rows via cached object: 11777674

In [4]:
# TODO: figure out a way to return to default outputs
# since verbose = TRUE is way too verbose compared to default
options(warn = 1) # Reenable warnings; see above comments


## Load claims files from GCS

In [5]:
# Define the target directory
target_directory <- "data-claims/raw/"

# Loop through the years 2018 to 2021
for (year in 2018:2021) {
  # Construct the file name
  file_name <- paste0("claims_extract_CLAIMS ", year, ".csv")
  bq_name <- paste0("claims_extract_CLAIMS\\ ", year, ".csv")

  # Check if the file exists in the target directory
  file_path <- here(target_directory, file_name)
  exists <- file.exists(file_path)

  # If the file does not exist, run the gsutil cp command
  if (!exists) {
    system(paste0("cd .. && gsutil cp gs://phic-claims-raw/", bq_name, " ", target_directory),
      intern = FALSE, ignore.stderr = FALSE
    )
  } else {
    message(paste("File", file_name, "already exists in the target directory. Skipping download."))
  }
}


File claims_extract_CLAIMS 2018.csv already exists in the target directory. Skipping download.

File claims_extract_CLAIMS 2019.csv already exists in the target directory. Skipping download.

File claims_extract_CLAIMS 2020.csv already exists in the target directory. Skipping download.

File claims_extract_CLAIMS 2021.csv already exists in the target directory. Skipping download.



## Load Mapping Data

In [6]:
# Read in all rvs codes and turn to character for further processing
# Run the gcloud bq query command to save the result as a CSV file
system(paste0(
  "bq query --use_legacy_sql=false --format=csv 'SELECT * FROM `drg-pipeline.grouper_v5.proc`' > ",
  here(aux_path, "proc.csv")
), intern = FALSE, ignore.stderr = FALSE)

# Read the CSV file into an R data frame
proc <- fread(here(aux_path, "proc.csv"))[, CODE := as.character(CODE)]

# Read in icd9cm equivalents of rvs codes,
# then convert to character and also remove decimals, whilst keeping trailing zeroes
system(paste0(
  "bq query --use_legacy_sql=false --format=csv 'SELECT * FROM `drg-pipeline.phic.acr_rvs_map`' > ",
  here(aux_path, "rvs_icd9cm.csv")
), intern = FALSE, ignore.stderr = FALSE)

rvs_icd9 <- fread(here(aux_path, "rvs_icd9cm.csv"), select = c("rvs", "icd9cm"))[, rvs := as.character(rvs)][, icd9cm := as.character(icd9cm * 100)]

# Merge with proc from above, to be able to classify by DRGUSE
rvs_icd9 <- merge(rvs_icd9, proc[, .(CODE, DRGUSE)], by.x = "icd9cm", by.y = "CODE", all.x = TRUE)

# Remove DRGUSE and filter out NAs
rvs_icd9 <- rvs_icd9[, is_drg := !is.na(DRGUSE) & DRGUSE][!is.na(rvs) & !is.na(icd9cm), -"DRGUSE"]

system(paste0(
  "bq query --use_legacy_sql=false --format=csv 'SELECT * FROM `drg-pipeline.phic.acr_procedure`' > ",
  here(aux_path, "acr_rvs.csv")
), intern = FALSE, ignore.stderr = FALSE)

# Read in PHIC all case rates
acr_rvs <- fread(here(aux_path, "acr_rvs.csv"))

system(paste0(
  "bq query --use_legacy_sql=false --format=csv 'SELECT * FROM `drg-pipeline.grouper_v5.i10`' > ",
  here(aux_path, "i10.csv")
), intern = FALSE, ignore.stderr = FALSE)

# Read in the thai icd10 library
tdrg_icd10 <- fread(here(aux_path, "i10.csv"))

# Set the key if not already set
setkey(tdrg_icd10, "CODE")

# Subset and assign the result to acc_pdx
acc_pdx <- unique(tdrg_icd10[ACCPDX == "Y", CODE])


## Read, Process, Export Data (Looping through all parts)

In [7]:
all_parts_summaries <- list() # initialize list for summaries
processing_times <- numeric(split_parts) # initialize list for ETA
dim_dt <- vector() # initialize vector for dt dimensions
ncores <- availableCores() # detect number of cores available for parallelization

# Define a codeblock to avoid repeating it twice when to_profvis is TRUE and again if FALSE
# Makes it easier to maintain as well, since we only need to modify one section instead of two
unified_block <- function() {
  # Start main execution logic
  split_and_save_parts() # Read, split, and save partial files

  # Start the parallelization session or remain sequential
  # TODO: mclapply (unix-only) might be faster than future_lapply
  strat <- if (.Platform$OS.type == "unix") multicore else multisession
  if (to_parallel) plan(strat, workers = ncores) else plan(sequential)
      
  # For each partial file in N (split_parts) files,
  for (part in 1:split_parts) {
    # Process the partial file with or without parallelization
    result <- process_part(
      part, ncores, to_view_checks,
      global_seed, tmp_nrow, rvs_icd9, tdrg_icd10,
      acc_pdx, to_parallel, to_write, to_group, to_sample
    )

    # Save partial summaries to a list
    all_parts_summaries[[part]] <- result$combined_summary

    # Save partial processing time to a list
    processing_times[part] <- result$processing_time

    # Print status update and ETA
    # - VS Code: Updates are shown after complete execution
    # - Positron: Updates are shown live
    # - JupyterLab: Updates are shown after complete execution
    print_status_update(part, split_parts, processing_times)

    if (part == 1) dim_dt <<- dim(result$dt)

    rm(result)
    gc()
  }

  # Summaries are consolidated from 5 split_parts * 15 chunks = 75 sub outputs
  print_summary_tables( # Print final summaries
    combine_parts_summaries(all_parts_summaries, tmp_nrow),
    end_nrow
  )

  plan(sequential) # end parallelization
  # End main execution logic
  if (to_debug) return(NULL) # debug
}

# Call the main function with or without profvis
if (to_profvis) saveWidget(profvis({unified_block()}), here(profvis_path)) else unified_block()


Status Update
Finished: Part 1 of 5
Elapsed: 6 seconds
ETA: 23 seconds
Status Update
Finished: Part 2 of 5
Elapsed: 11 seconds
ETA: 16 seconds
Status Update
Finished: Part 3 of 5
Elapsed: 16 seconds
ETA: 11 seconds
Status Update
Finished: Part 4 of 5
Elapsed: 21 seconds
ETA: 5 seconds
Status Update
Finished: Part 5 of 5
Elapsed: 26 seconds
ETA: 0 seconds


Rename Success:
 TRUE 



Table: ICD Replacements 1

|old_code |new_code | count|
|:--------|:--------|-----:|
|J18.92   |J1892    |  5921|
|A09.9    |A099     |  3144|
|N39.0    |N390     |  2654|
|A97.1    |A971     |  1514|
|K29.1    |K291     |  1127|
|J45.90   |J4590    |  1005|
|I10.1    |I101     |   977|
|I10.9    |I109     |   903|
|A97.0    |A970     |   859|
|P36.9    |P369     |   759|


Table: ICD Replacements 2

|old_code |new_code | count|
|:--------|:--------|-----:|
|I21.9    |I219     |    42|
|E86.1    |E861     |    29|
|I63.9    |I639     |    19|
|I61.9    |I619     |    11|
|I21.4    |I214     |    10|
|O75.8  

## Runtime Estimation

In [8]:
print_time_estimates() # Print time estimates along with estimate for full claims file


Time spent (total)               : 33.219 sec elapsed
Time spent (t/row) for 94.2k rows: 0.35 msec
Time (est) (total) for 11.8m rows: 69.21 min


## Debugging

In [9]:
# in case we want to run this cell independently:
source(here::here("data-cleaning/r_scripts", "11_debug-functions.R"))

# Consolidate all r_scripts scripts into everything.R; useful for debugging
concatenate_r_files(here::here("data-cleaning/r_scripts"), here::here("data-cleaning/everything/everything.R"))


In [10]:
rm(list = ls())
gc()


,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,1069771,57.2,2573510,137.5,2573510,137.5
Vcells,2039971,15.6,11845371,90.4,11844384,90.4


To extract all code portions of this ipynb file (run in VS Code terminal):

jupyter nbconvert --no-prompt --to script data-cleaning/drg-cleaning.ipynb --output everything/drg-cleaning

